# exp_10 Algorithm Comparison


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
from itertools import product

config = load_experiment_config()
train_df, val_df, _ = data_loading.load_berlin_splits(DATA_DIR / 'phase_3_experiments')

feature_cols = data_loading.get_feature_columns(train_df)

x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(
    train_df[feature_cols], x_val=val_df[feature_cols]
)

y_train, label_to_idx, idx_to_label = preprocessing.encode_genus_labels(train_df['genus_latin'])
y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()

groups = train_df['block_id'].values
cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])

results = []

# Baselines
majority = models.create_majority_classifier(y_train)
majority_preds = majority(x_val_scaled)
majority_metrics = evaluation.compute_metrics(y_val, majority_preds)
results.append({'algorithm': 'majority', 'type': 'baseline', 'val_f1_mean': majority_metrics['f1_score'], 'val_f1_std': 0.0, 'train_f1_mean': majority_metrics['f1_score'], 'train_val_gap': 0.0})

stratified = models.create_stratified_random_classifier(y_train)
strat_preds = stratified(x_val_scaled)
strat_metrics = evaluation.compute_metrics(y_val, strat_preds)
results.append({'algorithm': 'stratified_random', 'type': 'baseline', 'val_f1_mean': strat_metrics['f1_score'], 'val_f1_std': 0.0, 'train_f1_mean': strat_metrics['f1_score'], 'train_val_gap': 0.0})

# Spatial-only baseline (if coordinates available)
if {'x', 'y'} <= set(feature_cols):
    coord_indices = (feature_cols.index('x'), feature_cols.index('y'))
    spatial_model = models.create_spatial_only_rf(
        x_train_scaled, y_train, coord_indices=coord_indices
    )
    spatial_preds = spatial_model.predict(x_val_scaled[:, list(coord_indices)])
    spatial_metrics = evaluation.compute_metrics(y_val, spatial_preds)
    results.append({'algorithm': 'spatial_only_rf', 'type': 'baseline', 'val_f1_mean': spatial_metrics['f1_score'], 'val_f1_std': 0.0, 'train_f1_mean': spatial_metrics['f1_score'], 'train_val_gap': 0.0})

# ML models with coarse grid search
for model_name in ['random_forest', 'xgboost']:
    print(f"\nGrid search for {model_name}...")
    grid = config['algorithm_comparison']['models'][model_name]['grid']
    param_names = list(grid.keys())
    param_values = list(grid.values())
    
    best_val_f1 = -1
    best_params = {}
    best_metrics = {}
    
    n_configs = 1
    for vals in param_values:
        n_configs *= len(vals)
    print(f"  Testing {n_configs} configurations...")
    
    for i, combo in enumerate(product(*param_values), 1):
        params = dict(zip(param_names, combo))
        model = models.create_model(model_name, model_params=params)
        metrics = training.train_with_cv(model, x_train_scaled, y_train, groups, cv)
        
        if metrics['val_f1_mean'] > best_val_f1:
            best_val_f1 = metrics['val_f1_mean']
            best_params = params
            best_metrics = metrics
        
        if i % 10 == 0:
            print(f"    Config {i}/{n_configs}: best F1 = {best_val_f1:.4f}")
    
    print(f"  Best {model_name} params: {best_params}")
    print(f"  Best val F1: {best_val_f1:.4f}")
    
    results.append({
        'algorithm': model_name,
        'type': 'ml',
        'val_f1_mean': best_metrics['val_f1_mean'],
        'val_f1_std': best_metrics['val_f1_std'],
        'train_f1_mean': best_metrics['train_f1_mean'],
        'train_val_gap': best_metrics['train_val_gap'],
        'best_params': best_params,
    })

# NN models (optional)
try:
    temporal_features = [c for c in feature_cols if c.split('_')[-1].isdigit()]
    months = sorted({int(c.split('_')[-1]) for c in temporal_features})
    bases = sorted({"_".join(c.split('_')[:-1]) for c in temporal_features})
    n_temporal_bases = len(bases)
    n_months = len(months)
    n_static = len(feature_cols) - n_temporal_bases * n_months

    cnn_params = {
        'n_temporal_bases': n_temporal_bases,
        'n_months': n_months,
        'n_static_features': max(n_static, 0),
        'n_classes': len(label_to_idx),
    }
    cnn = models.create_model('cnn_1d', cnn_params)
    metrics = training.train_with_cv(cnn, x_train_scaled, y_train, groups, cv)
    results.append({
        'algorithm': 'cnn_1d',
        'type': 'nn',
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_f1_mean': metrics['train_f1_mean'],
        'train_val_gap': metrics['train_val_gap'],
    })
except Exception as exc:
    print(f"Skipping CNN1D: {exc}")

try:
    tabnet = models.create_model('tabnet')
    metrics = training.train_with_cv(tabnet, x_train_scaled, y_train, groups, cv)
    results.append({
        'algorithm': 'tabnet',
        'type': 'nn',
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_f1_mean': metrics['train_f1_mean'],
        'train_val_gap': metrics['train_val_gap'],
    })
except Exception as exc:
    print(f"Skipping TabNet: {exc}")

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_10_algorithm_comparison'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_algorithm_comparison(results_df, fig_dir / 'algorithm_comparison.png')
visualization.plot_train_val_gap(results_df, fig_dir / 'algorithm_train_val_gap.png')
visualization.plot_performance_ladder(
    results_df.rename(columns={'algorithm': 'name', 'val_f1_mean': 'f1_score'}).assign(category='model'),
    output_path=fig_dir / 'performance_ladder.png',
)

viable = results_df[(results_df['val_f1_mean'] >= 0.50) & (results_df['train_val_gap'] < 0.35)]
ml_candidates = viable[viable['type'] == 'ml']
nn_candidates = viable[viable['type'] == 'nn']
ml_champion = ml_candidates.sort_values('val_f1_mean', ascending=False).head(1).to_dict('records')
nn_champion = nn_candidates.sort_values('val_f1_mean', ascending=False).head(1).to_dict('records')

algo_path = OUTPUT_DIR / 'metadata/algorithm_comparison.json'
with algo_path.open('w') as handle:
    json.dump({
        'algorithms': results,
        'ml_champion': ml_champion[0] if ml_champion else {},
        'nn_champion': nn_champion[0] if nn_champion else {},
    }, handle, indent=2)
validate_algorithm_comparison(algo_path)

print(f"\nAlgorithm comparison complete")
print(f"ML Champion: {ml_champion[0]['algorithm'] if ml_champion else 'none'}")
print(f"NN Champion: {nn_champion[0]['algorithm'] if nn_champion else 'none'}")